# 01: Understanding SMILES Notation

Welcome to your first MolGen-Lab notebook! In this tutorial, we will learn how to represent and process molecular structures using **SMILES** (Simplified Molecular Input Line Entry System) strings and the **RDKit** library.

## 1. What is a SMILES string?

A SMILES string is a text representation of a molecule's structure. It encodes atoms, bonds, and rings into a single line of characters.

### Examples:
- **Water**: `O`
- **Ethanol**: `CCO` (two carbons and an oxygen)
- **Aspirin**: `CC(=O)Oc1ccccc1C(=O)O` (includes rings `c1...1` and branches `(...)`)
- **Caffeine**: `CN1C=NC2=C1C(=O)N(C(=O)N2C)C`

## 2. Installing and importing RDKit

RDKit is the powerhouse of chemoinformatics in Python. Let's import it and check its version.

In [1]:
import sys
print(sys.executable)

/Applications/Xcode.app/Contents/Developer/usr/bin/python3


In [ ]:
import rdkit
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import Descriptors

print(f"RDKit version: {rdkit.__version__}")

## 3. Parsing a SMILES string

We use `Chem.MolFromSmiles()` to convert a SMILES string into an internal RDKit molecule object.

In [ ]:
aspirin_smiles = "CC(=O)Oc1ccccc1C(=O)O"
mol = Chem.MolFromSmiles(aspirin_smiles)

if mol:
    print(f"Molecule parsed successfully!")
    print(f"Number of atoms: {mol.GetNumAtoms()}")
    print(f"Number of bonds: {mol.GetNumBonds()}")
else:
    print("Invalid SMILES string.")

## 4. Drawing a molecule

In a Jupyter notebook, RDKit can render molecules directly if they are the last item in a cell.

In [ ]:
# To render as an image object:
Draw.MolToImage(mol)

## 5. Validating SMILES

It's common to check if a string is a valid chemical structure before processing it.

In [ ]:
def is_valid_smiles(smiles: str) -> bool:
    """Checks if RDKit can successfully parse the SMILES string."""
    return Chem.MolFromSmiles(smiles) is not None

test_strings = [
    "CCO",         # Valid: Ethanol
    "c1ccccc1",    # Valid: Benzene
    "CN1C=NC2=C1C(=O)N(C(=O)N2C)C", # Valid: Caffeine
    "CCCCCX",      # Invalid: Atom 'X' doesn't exist
    "C1CCC",       # Invalid: Open ring (1 is not closed)
]

for s in test_strings:
    print(f"{s:30} -> Valid? {is_valid_smiles(s)}")

## 6. Molecular properties

Calculated properties can tell us a lot about a molecule's behavior.

In [ ]:
mw = Descriptors.MolWt(mol)
logp = Descriptors.MolLogP(mol)
hbd = Descriptors.NumHDonors(mol)
hba = Descriptors.NumHAcceptors(mol)

print(f"Aspirin Properties:")
print(f"- Molecular Weight: {mw:.2f} g/mol")
print(f"- LogP: {logp:.2f} (Lipophilicity)")
print(f"- H-Bond Donors: {hbd}")
print(f"- H-Bond Acceptors: {hba}")

## 7. Lipinski's Rule of 5

The Rule of 5 (Ro5) is a rule of thumb to evaluate druglikeness. A molecule is more likely to be orally active in humans if it meets these criteria:
1. Molecular weight < 500
2. LogP < 5
3. H-bond donors < 5
4. H-bond acceptors < 10

In [ ]:
def lipinski_pass(smiles: str) -> dict:
    mol = Chem.MolFromSmiles(smiles)
    if not mol: return {"error": "Invalid SMILES"}
    
    props = {
        "MW": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "HBD": Descriptors.NumHDonors(mol),
        "HBA": Descriptors.NumHAcceptors(mol)
    }
    
    # Logic for pass/fail
    passes = (
        props["MW"] < 500 and 
        props["LogP"] < 5 and 
        props["HBD"] < 5 and 
        props["HBA"] < 10
    )
    
    props["Passed"] = passes
    return props

print("Aspirin:", lipinski_pass("CC(=O)Oc1ccccc1C(=O)O"))
print("Caffeine:", lipinski_pass("CN1C=NC2=C1C(=O)N(C(=O)N2C)C"))

## 8. Your turn!

Try searching for your favorite molecule on [PubChem](https://pubchem.ncbi.nlm.nih.gov/). Find its **Canonical SMILES** string and use the functions we built to visualize it and check its properties!